In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import bessel, filtfilt
from scipy.special import erfcinv
import json
import os
import glob
from numpy.lib.stride_tricks import sliding_window_view
from tensorflow.keras import layers

# CONFIGURACIÓN
DATA_DIR = 'datasets' 
MODEL_PATH = 'teacher_soa_resnet.keras' 
SCALERS_PATH = 'scalers.json'

WINDOW_SIZE = 128
UI_SAMPLES = 2  
FEATURES = ['Output_P2', 'Bit_Rate', 'Beta_RC', 'Rango_Corr']
TARGET_COL = 'I_trim'
FINE_TUNE_PERCENT = 0.40 

# CAPA DE ATENCIÓN
@tf.keras.utils.register_keras_serializable()
class SimpleAttention(layers.Layer):
    def __init__(self, units, **kwargs):
        super(SimpleAttention, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.W = layers.Dense(self.units, activation='tanh')
        self.V = layers.Dense(1)
        super(SimpleAttention, self).build(input_shape)

    def call(self, inputs):
        score = self.V(self.W(inputs))
        attention_weights = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(attention_weights * inputs, axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config

# FUNCIONES AUXILIARES

def aplicar_filtro_bessel(signal, ui_samples, cutoff_ratio=0.75):
    """Aplicar filtro de Bessel para suavizar el ruido de alta frecuencia en transiciones"""
    nyquist = ui_samples / 2.0
    baud_rate = 1.0  
    corte_normalizado = (cutoff_ratio * baud_rate) / nyquist
    corte_normalizado = min(corte_normalizado, 0.99)
    
    b, a = bessel(N=4, Wn=corte_normalizado, btype='low', analog=False)
    return filtfilt(b, a, signal)

def unnormalize_col(scaled_data, scalers, col_name):
    v_min, v_max = scalers['min'][col_name], scalers['max'][col_name]
    return scaled_data * (v_max - v_min + 1e-7) + v_min

def sym2bits(s_arr):
    """Convertir arreglo de símbolos PAM-4 a bits"""
    b_msb = (s_arr == 2) | (s_arr == 3)
    b_lsb = (s_arr == 1) | (s_arr == 2)
    return np.column_stack((b_msb, b_lsb)).ravel()

def calcular_ber_optima_estadistica(y_true, y_pred, ui_samples=2):
    if np.array_equal(y_true, y_pred):
        th_norm = [0.25, 0.50, 0.75]
        umbrales_reales = [th * (y_pred.max() - y_pred.min()) + y_pred.min() for th in th_norm]
        return 0.0, 0, umbrales_reales

    y_t_n = (y_true - y_true.min()) / (y_true.max() - y_true.min() + 1e-7)
    y_p_n = (y_pred - y_pred.min()) / (y_pred.max() - y_pred.min() + 1e-7)

    best_metric = -np.inf
    best_ber = 1.0
    best_delay = 0
    best_ths_norm = [0.25, 0.50, 0.75] 

    for d in range(-15, 16):
        if d > 0:
            t_full, p_full = y_t_n[d:], y_p_n[:-d]
        elif d < 0:
            t_full, p_full = y_t_n[:d], y_p_n[-d:]
        else:
            t_full, p_full = y_t_n, y_p_n

        # Muestreo de fase independiente
        off_t = np.argmax([np.var(t_full[off::ui_samples]) for off in range(ui_samples)])
        off_p = np.argmax([np.var(p_full[off::ui_samples]) for off in range(ui_samples)])

        t = t_full[off_t::ui_samples]
        p = p_full[off_p::ui_samples]

        min_len = min(len(t), len(p))
        t = t[:min_len]
        p = p[:min_len]

        if len(t) < 50: continue

        sym_tx = np.digitize(t, [0.25, 0.50, 0.75])

        mu = np.zeros(4)
        sg = np.zeros(4)
        valid = True
        for s in range(4):
            mask = (sym_tx == s)
            if np.sum(mask) > 5:
                mu[s] = np.mean(p[mask])
                sg[s] = np.std(p[mask])
            else:
                valid = False
                break
        
        if not valid: continue

        order_est = np.argsort(mu)
        mu_s = mu[order_est]
        sg_s = sg[order_est]

        if any(sg_s <= 0): continue
    
        q12 = (mu_s[1] - mu_s[0]) / (sg_s[0] + sg_s[1] + 1e-9)
        q23 = (mu_s[2] - mu_s[1]) / (sg_s[1] + sg_s[2] + 1e-9)
        q34 = (mu_s[3] - mu_s[2]) / (sg_s[2] + sg_s[3] + 1e-9)
        metric = min(q12, q23, q34)

        if metric > best_metric:
            best_metric = metric
            best_delay = d
            
            th1 = (sg_s[0]*mu_s[1] + sg_s[1]*mu_s[0]) / (sg_s[0] + sg_s[1] + 1e-9)
            th2 = (sg_s[1]*mu_s[2] + sg_s[2]*mu_s[1]) / (sg_s[1] + sg_s[2] + 1e-9)
            th3 = (sg_s[2]*mu_s[3] + sg_s[3]*mu_s[2]) / (sg_s[2] + sg_s[3] + 1e-9)
            th_calc = sorted([th1, th2, th3])
            
            sym_p_raw = np.digitize(p, th_calc)
            sym_hat = np.zeros_like(sym_p_raw)
            for i in range(4):
                sym_hat[sym_p_raw == i] = order_est[i]
                
            best_ber = np.mean(sym2bits(sym_tx) != sym2bits(sym_hat))
            best_ths_norm = th_calc

    umbrales_reales = [th * (y_pred.max() - y_pred.min()) + y_pred.min() for th in best_ths_norm]
    return best_ber, best_delay, umbrales_reales

def calcular_q_metrics(ber, min_ber=1e-12):
    """Convertir BER a Factor Q"""
    ber_seguro = max(ber, min_ber)
    q_lin = np.sqrt(2) * erfcinv(2 * ber_seguro)
    q_db = 20 * np.log10(q_lin)
    return q_lin, q_db


# --- 4. MOTOR PRINCIPAL DE PROCESAMIENTO MASIVO ---
if __name__ == "__main__":
    print("Iniciando motor de procesamiento masivo...")

    # Carga de recursos y parches
    if not hasattr(tf.keras.layers.Dense, '_parche_aplicado'):
        original_dense_from_config = tf.keras.layers.Dense.from_config
        @classmethod
        def patched_dense_from_config(cls, config):
            if 'quantization_config' in config: del config['quantization_config']
            return original_dense_from_config(config)
        tf.keras.layers.Dense.from_config = patched_dense_from_config
        tf.keras.layers.Dense._parche_aplicado = True

    teacher_model = tf.keras.models.load_model(
        MODEL_PATH, compile=False, custom_objects={'SimpleAttention': SimpleAttention}
    )

    with open(SCALERS_PATH, 'r') as f:
        scalers = json.load(f)

    archivos_parquet = glob.glob(os.path.join(DATA_DIR, "*.parquet"))
    
    if not archivos_parquet:
        print(f"Error: No se encontraron archivos .parquet en la ruta: {DATA_DIR}")
        exit()
        
    print(f"Se encontraron {len(archivos_parquet)} archivos para procesar.")

    # Lista para acumular resultados del histograma
    resultados_delta_q = []

    # BUCLE DE PROCESAMIENTO
    for i, archivo in enumerate(archivos_parquet):
        print(f"\n[{i+1}/{len(archivos_parquet)}] Procesando: {os.path.basename(archivo)}")
        
        try:
            df = pd.read_parquet(archivo)

            x_scaled = np.column_stack([
                (df[col].values.astype(np.float32) - scalers['min'][col]) / (scalers['max'][col] - scalers['min'][col] + 1e-7)
                for col in FEATURES
            ])
            y_real_scaled = (df[TARGET_COL].values.astype(np.float32) - scalers['min'][TARGET_COL]) / (scalers['max'][TARGET_COL] - scalers['min'][TARGET_COL] + 1e-7)

            x_windows = sliding_window_view(x_scaled[:-1], (WINDOW_SIZE, len(FEATURES)))
            x_windows = x_windows.reshape(-1, WINDOW_SIZE, len(FEATURES))

            offset = WINDOW_SIZE // 2
            y_windows_aligned = y_real_scaled[offset : offset + len(x_windows)]

            split_idx = int(len(x_windows) * FINE_TUNE_PERCENT)
            x_calib, y_calib = x_windows[:split_idx], y_windows_aligned[:split_idx]
            x_infer = x_windows[split_idx:]

            # --- Fine-Tuning  ---
            for layer in teacher_model.layers[:-2]: layer.trainable = False
            teacher_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
            early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
            
            print(f"   ⏳ Entrenando IA (Fine-tuning)...")
            teacher_model.fit(x_calib, y_calib, epochs=10, batch_size=256, validation_split=0.15, callbacks=[early_stop], verbose=0)
            
            # --- Inferencia  ---
            y_pred_scaled = teacher_model.predict(x_infer, batch_size=1024, verbose=0).flatten()
            prediccion_final_cruda = unnormalize_col(y_pred_scaled, scalers, TARGET_COL)
            prediccion_final = aplicar_filtro_bessel(prediccion_final_cruda, UI_SAMPLES, cutoff_ratio=0.85)

            # --- Evaluación ---
            start_infer_idx = offset + split_idx
            end_infer_idx = start_infer_idx + len(prediccion_final)
            ideal_alineado = df['I_trim'].values[start_infer_idx : end_infer_idx]
            distorsionado_alineado = df['Output_P2'].values[start_infer_idx : end_infer_idx]

            ber_soa, _, _ = calcular_ber_optima_estadistica(ideal_alineado, distorsionado_alineado, UI_SAMPLES)
            ber_ia, _, _ = calcular_ber_optima_estadistica(ideal_alineado, prediccion_final, UI_SAMPLES)

            q_lin_soa, q_db_soa = calcular_q_metrics(ber_soa)
            q_lin_ia, q_db_ia = calcular_q_metrics(ber_ia)

            delta_q_db = q_db_ia - q_db_soa
            resultados_delta_q.append(delta_q_db)
            
            print(f"    Sin Ecualizador (S/EQ) -> BER: {ber_soa:.6f} | Q: {q_lin_soa:.2f} | Q[dB]: {q_db_soa:.2f} dB")
            print(f"    Con Ecualizador (C/EQ) -> BER: {ber_ia:.6f} | Q: {q_lin_ia:.2f} | Q[dB]: {q_db_ia:.2f} dB")
            print(f"    ΔQ: {delta_q_db:+.2f} dB")

        except Exception as e:
            print(f"    Error procesando {os.path.basename(archivo)}: {e}")

    # HISTOGRAMA
    print("\n" + "="*50)
    print(" GENERANDO REPORTE FINAL...")
    print("="*50)

    if resultados_delta_q:
        plt.figure(figsize=(10, 6), facecolor='white')
        
        # Histograma
        n, bins, patches = plt.hist(resultados_delta_q, bins=30, color='#2ca02c', alpha=0.7, edgecolor='black')
        
        plt.axvline(0, color='red', linestyle='--', linewidth=2, label='0 dB (Sin Mejora)')
        media_delta = np.mean(resultados_delta_q)
        plt.axvline(media_delta, color='blue', linestyle='-.', linewidth=2, label=f'Media: {media_delta:+.2f} dB')

        plt.title('Distribución de Mejora del Factor Q ($\Delta Q_{dB}$)', fontsize=14, fontweight='bold')
        plt.xlabel('$\Delta Q = Q_{C/EQ} - Q_{S/EQ}$ [dB]', fontsize=12)
        plt.ylabel('Frecuencia (Casos)', fontsize=12)
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend()
        
        for i, patch in enumerate(patches):
            if bins[i] < 0:
                patch.set_facecolor('#d62728') 
        
        plt.tight_layout()
        plt.show()
        
        print(f"Proceso masivo completado exitosamente. Mejora media global: {media_delta:+.2f} dB")
    else:
        print("No se generaron resultados válidos para graficar.")